# 5685: semantic embedding initialisation for Delphi-2M

Does initialising Delphi-2M's token embeddings from language-model representations of the ICD-10 code descriptions improve a generative disease-trajectory model, and does any gain concentrate on rare codes? This is the generative analogue of the GRASP question.

The design is three initialisation arms (random baseline, statistics-matched control, semantic) crossed with two output-embedding conditions (tied, untied), three seeds each, 18 runs. Delphi ties the token embedding `wte` to `lm_head`, so a semantic initialisation is both an input representation and an output prior. Untying separates the two and tests whether tying is what suppresses any effect.

Run order: 1 setup, 2 descriptions, 3 embeddings, 4 sanity check, 5 projection, 6 training, 7 evaluation.

Runtime: T4 GPU. Do not run `pip install -r requirements.txt`; it downgrades numpy and breaks the Colab runtime.

Two implementation details that matter. `configure_optimizers` in the original code removes `lm_head.weight` from the decay set unconditionally; in the untied model that parameter is real, so the removal is guarded. `tie_weights` is written into `model_args`, because a checkpoint reloaded into a tied model would otherwise overwrite `wte` with `lm_head` on load and silently evaluate the wrong model. Both were verified by round-tripping a checkpoint in each condition.

## 1. Setup

In [ ]:
import torch, numpy as np, pandas as pd, sys, os
print("python", sys.version.split()[0])
print("torch ", torch.__version__, "| cuda:", torch.cuda.is_available())
print("numpy ", np.__version__)
assert torch.cuda.is_available(), "Switch to a GPU runtime (Runtime > Change runtime type > T4 GPU)"

python 3.12.13
torch  2.11.0+cu128 | cuda: True
numpy  2.0.2


In [ ]:
IN_COLAB = 'google.colab' in sys.modules
DRIVE = '/content/drive/MyDrive/delphi_5685'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE, exist_ok=True)
else:
    DRIVE = os.path.abspath('./delphi_5685'); os.makedirs(DRIVE, exist_ok=True)
print("artifacts ->", DRIVE)

import shutil

def keep(*paths):
    'Copy files or run directories to persistent storage, overwriting cleanly.'
    for p in paths:
        if not os.path.exists(p):
            continue
        dst = os.path.join(DRIVE, os.path.basename(p.rstrip('/')))
        if os.path.isdir(p):
            shutil.rmtree(dst, ignore_errors=True)   # plain `cp -r` would nest dir-in-dir
            shutil.copytree(p, dst)
        else:
            shutil.copy2(p, dst)
        print("saved ->", dst)

def restore(name):
    'Bring a run directory back from Drive after a runtime reset.'
    src = os.path.join(DRIVE, name)
    if os.path.isdir(src) and not os.path.exists(f'{name}/ckpt.pt'):
        shutil.rmtree(name, ignore_errors=True)
        shutil.copytree(src, name)
        print("restored", name, "from Drive")

def restore_file(*names):
    'Bring saved artifacts back from Drive. True only if every one is now present.'
    for n in names:
        src = os.path.join(DRIVE, n)
        if os.path.isfile(src) and not os.path.isfile(n):
            shutil.copy2(src, n); print("restored", n, "from Drive")
    return all(os.path.isfile(n) for n in names)

Mounted at /content/drive
artifacts -> /content/drive/MyDrive/delphi_5685


In [ ]:
ROOT = '/content' if IN_COLAB else os.getcwd()
%cd $ROOT
!test -d Delphi || git clone -q https://github.com/gerstung-lab/Delphi.git
%cd Delphi
# nothing to install: torch, numpy, pandas, sklearn and transformers are already in Colab.
import numpy as np; print("numpy still ok:", np.__version__)

/content
/content/Delphi
numpy still ok: 2.0.2


## 2. Code descriptions

Delphi-2M's vocabulary is 1,270 tokens: 0 padding, 1 "no event", 2 to 3 sex, 4 to 12 BMI, smoking and alcohol strata, 13 to 1268 ICD-10 three-character codes, 1269 death.

Two choices for the embedding text. Each code is described with its ICD-10 chapter appended ("Cholera. ICD-10 code A00, Certain infectious and parasitic diseases"), since the chapter is the only extra supervision the label file provides. The non-disease tokens get hand-written descriptions rather than their literal labels, so that "Padding" does not land next to "No event" for reasons unrelated to physiology. Token 0 is overwritten with the model's own random initialisation in section 5, as it is masked out of attention and listed in `ignore_tokens`.

In [ ]:
import re
lab = pd.read_csv('delphi_labels_chapters_colours_icd.csv')
V = len(lab)
assert V == 1270 and (lab['index'].values == np.arange(V)).all(), "label file changed upstream"

MANUAL = {
    0:  "Padding token. Not a clinical concept.",
    1:  "No event recorded during this interval of follow-up. The person remained free of any "
        "new recorded diagnosis.",
    2:  "Female sex, recorded at study baseline.",
    3:  "Male sex, recorded at study baseline.",
    4:  "Low body mass index. Underweight or lean body habitus.",
    5:  "Mid body mass index. Normal or moderately overweight body habitus.",
    6:  "High body mass index. Obesity, a risk factor for metabolic and cardiovascular disease.",
    7:  "Low tobacco smoking exposure. Never or minimal smoker.",
    8:  "Moderate tobacco smoking exposure.",
    9:  "Heavy tobacco smoking exposure. A major risk factor for lung cancer, chronic obstructive "
        "pulmonary disease and vascular disease.",
    10: "Low alcohol consumption.",
    11: "Moderate alcohol consumption.",
    12: "Heavy alcohol consumption. A risk factor for liver disease, some cancers and injury.",
}

ICD = re.compile(r'^([A-Z][0-9X]{2})\s+(.*)$')   # [0-9X] also catches 'CXX Unknown Cancer'

def describe(i, name, chapter):
    if i in MANUAL:
        return MANUAL[i]
    m = ICD.match(str(name))
    if m:
        code_, desc = m.group(1), m.group(2)
        return f"{desc}. ICD-10 code {code_}, {chapter}."
    return f"{name}. {chapter}."      # 'Death'

texts = [describe(i, n, c) for i, (n, c) in enumerate(zip(lab.name, lab['ICD-10 Chapter']))]
assert len(texts) == V
for i in (0, 1, 6, 13, 499, 1142, 1269):
    print(f"{i:>4}  {texts[i]}")

   0  Padding token. Not a clinical concept.
   1  No event recorded during this interval of follow-up. The person remained free of any new recorded diagnosis.
   6  High body mass index. Obesity, a risk factor for metabolic and cardiovascular disease.
  13  Cholera. ICD-10 code A00, I. Certain infectious and parasitic diseases.
 499  Essential primary hypertension. ICD-10 code I10, IX. Diseases of the circulatory system.
1142  Unknown Cancer. ICD-10 code CXX, II. Neoplasms.
1269  Death. Death.


## 3. Embeddings

`BACKEND = 'local'` runs a PubMedBERT sentence encoder on the GPU. It needs no API key, takes under a minute for 1,270 short strings, and a reviewer can reproduce the exact vectors. This is the default.

`BACKEND = 'gemini'` is kept for comparing encoder families. The loop uses exponential backoff, resumes on a batch boundary, and requests 768 dimensions directly via Matryoshka truncation, re-normalised as the API requires below 3,072.

In [ ]:
BACKEND    = 'local'                              # 'local' | 'gemini'
LOCAL_MODEL = 'NeuML/pubmedbert-base-embeddings'  # PubMed-trained, mean-pooled, 768-d
RAW = f'semantic_raw_{BACKEND}.npy'

REUSE_RAW = restore_file(RAW)
print("reusing embeddings from Drive" if REUSE_RAW else "no saved embeddings — will encode")

restored semantic_raw_local.npy from Drive
reusing embeddings from Drive


In [ ]:
if BACKEND == 'local' and not REUSE_RAW:
    from transformers import AutoTokenizer, AutoModel

    tok = AutoTokenizer.from_pretrained(LOCAL_MODEL)
    enc = AutoModel.from_pretrained(LOCAL_MODEL).eval().cuda()

    @torch.no_grad()
    def embed(batch):
        t = tok(batch, padding=True, truncation=True, max_length=64, return_tensors='pt')
        t = {k: v.cuda() for k, v in t.items()}
        h = enc(**t).last_hidden_state                      # (B, T, 768)
        m = t['attention_mask'].unsqueeze(-1).float()
        return ((h * m).sum(1) / m.sum(1)).cpu().numpy()    # mean pooling

    E = np.concatenate([embed(texts[i:i+64]) for i in range(0, len(texts), 64)]).astype(np.float32)
    np.save(RAW, E)
    print("embedded", E.shape, "| per-row norm mean", float(np.linalg.norm(E, axis=1).mean()).__round__(3))
    del enc; torch.cuda.empty_cache()

In [ ]:
if BACKEND == 'gemini' and not REUSE_RAW:
    import time
    from getpass import getpass
    !pip install -q google-genai
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=getpass('Gemini API key: '))
    EMB, DIM, B = "models/gemini-embedding-001", 768, 50
    PART = f'{DRIVE}/semantic_raw_partial.npy'

    vecs = [list(v) for v in np.load(PART)] if os.path.exists(PART) else []
    start = (len(vecs) // B) * B           # drop a partial batch, resume on a boundary
    vecs = vecs[:start]
    print(f"resuming at {start} of {len(texts)}")

    cfg = types.EmbedContentConfig(task_type='SEMANTIC_SIMILARITY', output_dimensionality=DIM)
    for i in range(start, len(texts), B):
        for attempt in range(6):
            try:
                r = client.models.embed_content(model=EMB, contents=texts[i:i+B], config=cfg)
                vecs.extend([e.values for e in r.embeddings])
                break
            except Exception as e:
                msg = str(e)
                if 'PerDay' in msg:                       # daily cap: no backoff will help
                    np.save(PART, np.array(vecs, dtype=np.float32))
                    raise RuntimeError(f"daily quota exhausted at {len(vecs)}/{len(texts)}; "
                                       f"rerun tomorrow, it resumes from here. {msg[:200]}")
                if attempt == 5:
                    np.save(PART, np.array(vecs, dtype=np.float32))
                    raise RuntimeError(f"gave up at {len(vecs)}/{len(texts)}: {msg[:200]}")
                time.sleep(min(60, 2 ** attempt * 5))     # 5,10,20,40,60,60 s
        np.save(PART, np.array(vecs, dtype=np.float32))
        print(i, end=' ', flush=True)

    E = np.array(vecs, dtype=np.float32)
    E /= np.linalg.norm(E, axis=1, keepdims=True)   # required when output_dimensionality < 3072
    assert E.shape[0] == len(texts), f"incomplete: {E.shape[0]} of {len(texts)}"
    np.save(RAW, E)
    print("\nembedded", E.shape)

In [ ]:
E = np.load(RAW).astype(np.float32)
assert E.shape[0] == V, f"{E.shape[0]} embeddings for {V} tokens"
assert np.isfinite(E).all()
keep(RAW)
print("E:", E.shape, "dtype", E.dtype)

saved -> /content/drive/MyDrive/delphi_5685/semantic_raw_local.npy
E: (1270, 768) dtype float32


## 4. Sanity check: do the embeddings carry clinical structure?

Two checks. The nearest-neighbour probe is the readable one; the chapter-purity number is the one to quote. Purity is the fraction of a code's 10 nearest neighbours that share its ICD-10 chapter, averaged over the 1,256 disease codes, against a null obtained by shuffling the chapter labels. If purity is not clearly above the null there is no signal to transfer and the rest of the notebook is not worth running.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
S = cosine_similarity(E)
np.fill_diagonal(S, -np.inf)

for probe in ['E11', 'C50', 'I21', 'F32', 'J45']:
    hits = lab.index[lab.name.str.startswith(probe)]
    if not len(hits):
        continue
    i = int(hits[0])
    print(f"\n{lab.name[i]}")
    for j in np.argsort(-S[i])[:5]:
        print(f"    {S[i, j]:.3f}  {lab.name[j]}")


E11 Non-insulin-dependent diabetes mellitus
    0.979  E10 Insulin-dependent diabetes mellitus
    0.938  E12 Malnutrition-related diabetes mellitus
    0.935  E13 Other specified diabetes mellitus
    0.926  E14 Unspecified diabetes mellitus
    0.867  E15 Nondiabetic hypoglycaemic coma

C50 Malignant neoplasm of breast
    0.944  C56 Malignant neoplasm of ovary
    0.937  C52 Malignant neoplasm of vagina
    0.932  C71 Malignant neoplasm of brain
    0.926  C67 Malignant neoplasm of bladder
    0.919  C00 Malignant neoplasm of lip

I21 Acute myocardial infarction
    0.977  I22 Subsequent myocardial infarction
    0.935  I23 Certain current complications following acute myocardial infarction
    0.915  I40 Acute myocarditis
    0.913  I24 Other acute ischaemic heart diseases
    0.911  I42 Cardiomyopathy

F32 Depressive episode
    0.978  F33 Recurrent depressive disorder
    0.961  F30 Manic episode
    0.948  F38 Other mood [affective] disorders
    0.947  F31 Bipolar affective di

In [ ]:
K = 10
dis = np.arange(V) > 12                                  # disease codes + death only
chap = lab['ICD-10 Chapter'].values
nbr = np.argsort(-S[np.ix_(dis, np.arange(V))], axis=1)[:, :K]
purity = (chap[nbr] == chap[dis][:, None]).mean()

rng = np.random.default_rng(0)
def shuffled():
    p = rng.permutation(chap)                    # one relabelling, applied consistently
    return (p[nbr] == p[dis][:, None]).mean()
null = float(np.mean([shuffled() for _ in range(50)]))

print(f"chapter purity @{K}: {purity:.3f}   shuffled null: {null:.3f}   ratio: {purity/null:.1f}x")
assert purity > 2 * null, "embeddings carry little chapter structure — stop and change encoder"

chapter purity @10: 0.993   shuffled null: 0.067   ratio: 14.9x


## 5. Project to the model dimension

Delphi-2M uses `n_embd = 120`. PCA to 120 dimensions, project each row onto the unit sphere so that cosine geometry is what survives and no code gets a large-norm head start in the tied output layer, then apply one global scalar so the per-element standard deviation is exactly 0.02, the value `Delphi._init_weights` uses for `nn.Embedding`. Matching the scale matters: an initialisation with a different scale changes early-training dynamics on its own, and any baseline-versus-semantic difference would be unattributable.

The control is Gaussian noise put through the identical normalise-and-scale pipeline: same shape, same row norms, same standard deviation, no meaning. Semantic minus control is the effect of meaning; control minus baseline is the effect of the pipeline's geometry.

In [ ]:
import hashlib
from sklearn.decomposition import PCA
N_EMBD, INIT_STD = 120, 0.02

def to_init(X):
    X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)   # onto the sphere
    return (X * (INIT_STD / X.std())).astype(np.float32)        # one global scalar

# The tied runs were trained on specific .npy files. The untied runs MUST use the same
# bytes, or the two conditions are not comparable and the interaction test is meaningless.
# Regenerating would *probably* reproduce them (PCA is seeded, the control RNG is seeded)
# but "probably" is not a thing to build a result on across library versions.
if restore_file('semantic_init.npy', 'control_init.npy'):
    E120, Ectrl, PCA_VAR = np.load('semantic_init.npy'), np.load('control_init.npy'), None
    print("reusing the initialisations the tied runs were trained on")
else:
    pca  = PCA(n_components=N_EMBD, random_state=0)
    E120 = to_init(pca.fit_transform(E))
    PCA_VAR = float(pca.explained_variance_ratio_.sum())
    rng   = np.random.default_rng(0)
    Ectrl = to_init(rng.standard_normal((V, N_EMBD)))
    # token 0 is padding: masked out of attention and in ignore_tokens, so it must carry
    # no semantics. Give both arms the same neutral row.
    pad = rng.standard_normal(N_EMBD).astype(np.float32) * INIT_STD
    E120[0] = pad; Ectrl[0] = pad
    np.save('semantic_init.npy', E120); np.save('control_init.npy', Ectrl)
    keep('semantic_init.npy', 'control_init.npy')
    print(f"generated fresh initialisations  (PCA var explained {PCA_VAR:.3f})")

# these hold whichever branch ran
assert E120.shape == (V, N_EMBD) and Ectrl.shape == (V, N_EMBD), "wrong shape for this model"
assert abs(E120.std() - Ectrl.std()) < 1e-4, "arms differ in scale — the comparison is confounded"
assert abs(E120.std() - INIT_STD) < 1e-4, "init scale does not match Delphi._init_weights"

SHA = {f: hashlib.sha256(open(f, 'rb').read()).hexdigest()[:16]
       for f in ('semantic_init.npy', 'control_init.npy')}
for f, h in SHA.items():
    print(f"  {f:20s} std {np.load(f).std():.4f}  sha256:{h}")
print("\nquote these hashes in the write-up: they prove both conditions used identical inits")

restored semantic_init.npy from Drive
restored control_init.npy from Drive
reusing the initialisations the tied runs were trained on
  semantic_init.npy    std 0.0200  sha256:4321d995ef111738
  control_init.npy     std 0.0200  sha256:e563b8eccbdb4587

quote these hashes in the write-up: they prove both conditions used identical inits


## 6. Training: a 3 x 2 design

Three initialisation arms, identical in every respect except `wte`:

| arm | `wte` initialisation |
|---|---|
| `baseline` | nanoGPT default, N(0, 0.02^2) |
| `control` | random, matched norms and standard deviation |
| `sem` | PubMedBERT, PCA, matched norms and standard deviation |

crossed with weight tying:

| condition | `wte` and `lm_head` |
|---|---|
| `tied` | shared, as Delphi is published |
| `untied` | separate `lm_head`, so `wte` is only an input representation |

Why untying. In Delphi, `wte` doubles as the decoder, so a semantic initialisation sets both the geometry the encoder reads inputs through and the output logit directions. Those want different things: a geometry where asthma and COPD sit close is good for representing an input, but it also forces the model to assign them near-identical next-token probabilities before it has seen an example. That is a coherent explanation for a null result, and it is testable with one flag.

One confound to state. Untying adds vocab_size x n_embd = 152,400 parameters (2.24M to 2.40M) and removes the regularisation that tying imposes, so untied versus tied is not a clean comparison. The comparison that matters is within each condition, whether the semantic arm's delta is negative and concentrated in the rare bins, and whether that pattern differs between conditions.

Architecture is otherwise Delphi-2M as published (`config/train_delphi_demo.py`). Data is the authors' synthetic release, usable for the mechanism question and not for a clinical claim. About 10 minutes per run on a T4; three seeds x three arms x two conditions = 18 runs.

In [ ]:
# ---------- model.py -> model_untied.py : make weight tying a config flag ----------
msrc = open('model.py').read()

a = "    mask_ties: bool = False"
assert msrc.count(a) == 1, "model.py changed upstream — re-check the anchor"
msrc = msrc.replace(a, a + "\n    tie_weights: bool = True", 1)

a = ("        self.transformer.wte.weight = self.lm_head.weight"
     " # https://paperswithcode.com/method/weight-tying")
assert msrc.count(a) == 1, "model.py changed upstream — re-check the anchor"
msrc = msrc.replace(a, "        if config.tie_weights:\n    " + a, 1)

# Untied, lm_head.weight is a real parameter and must STAY in the decay group. Left
# unguarded, configure_optimizers drops it from `decay` and its own completeness
# assert fires.
a = "        decay.remove('lm_head.weight')"
assert msrc.count(a) == 1, "model.py changed upstream — re-check the anchor"
msrc = msrc.replace(a, "        if self.config.tie_weights:\n    " + a, 1)

open('model_untied.py', 'w').write(msrc)
import py_compile; py_compile.compile('model_untied.py', doraise=True)
print("wrote model_untied.py")

wrote model_untied.py


In [ ]:
# ---------- train.py -> train_sem.py ----------
src = open('train.py').read()

a = "from model import Delphi, DelphiConfig"
assert src.count(a) == 1
src = src.replace(a, "from model_untied import Delphi, DelphiConfig", 1)

a = "mask_ties = True"
assert src.count(a) == 1
src = src.replace(a, a + "\ntie_weights = True", 1)

# tie_weights MUST reach model_args. Without it a checkpoint reloads into a *tied*
# model, and load_state_dict then writes wte and lm_head into the same storage —
# lm_head wins, wte is silently destroyed, and evaluation reports nonsense.
a = "                  mask_ties=mask_ties, ignore_tokens=ignore_tokens)"
assert src.count(a) == 1
src = src.replace(a, "                  mask_ties=mask_ties, ignore_tokens=ignore_tokens,\n"
                     "                  tie_weights=tie_weights)", 1)

# (a) declare the knobs before configurator.py execs, so --init_embeddings=... is accepted
a1 = "init_from = 'scratch'"
assert src.count(a1) == 1, "train.py changed upstream — re-check the anchor"
src = src.replace(a1, a1 + "\ninit_embeddings = ''\nfreeze_embeddings = False", 1)

# (b) inject at MODULE level. v1 anchored on `model = Delphi(gptconf)`, which sits indented
#     inside `if init_from == 'scratch':` — the injected block landed between that `if` and its
#     `elif init_from == 'resume':`, silently re-binding the elif. `model.to(device)` is unique
#     and unindented.
a2 = "model.to(device)"
assert src.count(a2) == 1, "train.py changed upstream — re-check the anchor"
src = src.replace(a2, '''# --- semantic embedding initialisation ------------------------------------
if init_embeddings:
    _E = np.load(init_embeddings)
    _W = model.transformer.wte.weight
    assert _E.shape == tuple(_W.shape), f"{_E.shape} vs {tuple(_W.shape)}"
    with torch.no_grad():
        _W.copy_(torch.from_numpy(_E).to(_W.dtype))
    print(f"[init] wte <- {init_embeddings}  std={_E.std():.4f}  "
          f"tied_to_lm_head={_W is model.lm_head.weight}")
    if freeze_embeddings:
        _W.requires_grad_(False)
        print("[init] wte FROZEN (tied => lm_head frozen too)")
# --------------------------------------------------------------------------

model.to(device)''', 1)

open('train_sem.py', 'w').write(src)
py_compile.compile('train_sem.py', doraise=True)
print("wrote train_sem.py")

wrote train_sem.py


In [ ]:
# sanity: both conditions build, and the optimizer still covers every parameter
from model_untied import Delphi, DelphiConfig
for tie in (True, False):
    m = Delphi(DelphiConfig(vocab_size=1270, n_embd=120, n_layer=12, n_head=12,
                            block_size=96, tie_weights=tie))
    n_opt = sum(len(g['params'])
                for g in m.configure_optimizers(0.2, 2e-3, (0.9, 0.99), 'cpu').param_groups)
    print(f"tie_weights={tie!s:5s}  shared={m.transformer.wte.weight is m.lm_head.weight!s:5s}  "
          f"optimizer covers {n_opt}/{len(list(m.parameters()))} tensors")
    assert n_opt == len(list(m.parameters()))
    del m

number of parameters: 2.26M
using fused AdamW: False
tie_weights=True   shared=True   optimizer covers 148/148 tensors
number of parameters: 2.41M
using fused AdamW: False
tie_weights=False  shared=False  optimizer covers 149/149 tensors


### Reusing earlier runs

The cell below reuses anything already trained: it renames `run_{arm}_s{seed}` to `run_{arm}_tied_s{seed}`, restores from Drive if the local directory is gone, then skips any run that already has a `ckpt.pt`. Only the missing runs train. A checkpoint saved before the `tie_weights` key existed loads correctly, since `DelphiConfig` defaults it to `True`; verified by round-trip, with `wte` coming back bit-identical.

`BACKUP_RUNS = True` copies each finished run back to Drive. At about 28 MB per run the full 18-run grid is about 500 MB.

The initialisations are restored from Drive rather than regenerated, and their SHA-256 hashes are printed. The tied runs were trained on those exact bytes; the untied runs must use the same ones or the interaction test compares two different experiments. Both conditions read the same `SEEDS` list so the grid stays balanced.

In [ ]:
import subprocess, time

SEEDS = [0, 1, 2]                  # all three v2 tied runs survived, so use all three
BACKUP_RUNS = True                   # copy each finished run to Drive (~28 MB each)
ARMS  = {'baseline': [], 'control': ['--init_embeddings=control_init.npy'],
         'sem':      ['--init_embeddings=semantic_init.npy']}
TIES  = {'tied': 'True', 'untied': 'False'}
RUNS  = {t: {a: [] for a in ARMS} for t in TIES}

# v2 named the tied runs run_{arm}_s{seed}. Adopt them instead of retraining: their
# model_args predate the tie_weights key, and DelphiConfig defaults it to True, which
# is exactly what they were. Rename in Drive too, so restore() finds them.
for arm in ARMS:
    for s in SEEDS:
        for base in ('.', DRIVE):
            old, new = f'{base}/run_{arm}_s{s}', f'{base}/run_{arm}_tied_s{s}'
            if os.path.isdir(old) and not os.path.isdir(new):
                os.rename(old, new); print("adopted", old, "->", new)

for tie, flag in TIES.items():
    for arm, extra in ARMS.items():
        for s in SEEDS:
            out = f'run_{arm}_{tie}_s{s}'
            RUNS[tie][arm].append(out)
            restore(out)                            # survives a runtime reset
            if os.path.exists(f'{out}/ckpt.pt'):
                print("skip (exists)", out)
                if BACKUP_RUNS and not os.path.isdir(os.path.join(DRIVE, out)):
                    keep(out)                       # adopted v2 runs are not in Drive yet
                continue
            cmd = [sys.executable, 'train_sem.py', 'config/train_delphi_demo.py',
                   '--device=cuda', f'--out_dir={out}', f'--seed={s}',
                   f'--tie_weights={flag}'] + extra
            print("\n>>>", ' '.join(cmd), flush=True); t = time.time()
            p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in p.stdout:                   # stream, ~10 min of silence otherwise
                if line.startswith(('step ', '[init]', 'number of parameters', 'Traceback')) \
                   or 'Error' in line:
                    print("   ", line.rstrip(), flush=True)
            if p.wait() != 0:
                raise RuntimeError(f"{out} failed — rerun the command above unfiltered to see why")
            print(f"    done in {time.time()-t:.0f}s")
            if BACKUP_RUNS:
                keep(out)

print("\nruns:"); [print(" ", t, a, d) for t, x in RUNS.items() for a, d in x.items()]

adopted /content/drive/MyDrive/delphi_5685/run_baseline_s0 -> /content/drive/MyDrive/delphi_5685/run_baseline_tied_s0
adopted /content/drive/MyDrive/delphi_5685/run_baseline_s1 -> /content/drive/MyDrive/delphi_5685/run_baseline_tied_s1
adopted /content/drive/MyDrive/delphi_5685/run_baseline_s2 -> /content/drive/MyDrive/delphi_5685/run_baseline_tied_s2
adopted /content/drive/MyDrive/delphi_5685/run_control_s0 -> /content/drive/MyDrive/delphi_5685/run_control_tied_s0
adopted /content/drive/MyDrive/delphi_5685/run_control_s1 -> /content/drive/MyDrive/delphi_5685/run_control_tied_s1
adopted /content/drive/MyDrive/delphi_5685/run_control_s2 -> /content/drive/MyDrive/delphi_5685/run_control_tied_s2
adopted /content/drive/MyDrive/delphi_5685/run_sem_s0 -> /content/drive/MyDrive/delphi_5685/run_sem_tied_s0
adopted /content/drive/MyDrive/delphi_5685/run_sem_s1 -> /content/drive/MyDrive/delphi_5685/run_sem_tied_s1
adopted /content/drive/MyDrive/delphi_5685/run_sem_s2 -> /content/drive/MyDrive/de

[None, None, None, None, None, None]

### Frozen embeddings, tied: why there is no frozen arm

Freezing `wte` in the tied model also freezes the decoder, so the model can only fit the time-to-event head. That would be a different model, not an initialisation experiment, which is why the comparison above has no frozen arm. One short run documents this.

Untying does make a frozen arm coherent (frozen input representation, learned decoder), and it is the natural experiment after this one. It is not run here: this notebook tests a single pre-specified manipulation.

In [ ]:
restore('run_frozen_demo')
if os.path.exists('run_frozen_demo/ckpt.pt'):
    print("run_frozen_demo already exists — skipping")
else:
    !python train_sem.py config/train_delphi_demo.py --device=cuda --out_dir=run_frozen_demo \
        --init_embeddings=semantic_init.npy --freeze_embeddings=True --tie_weights=True \
        --max_iters=500 --lr_decay_iters=500 --eval_interval=100 --always_save_checkpoint=True
    keep('run_frozen_demo')

restored run_frozen_demo from Drive
run_frozen_demo already exists — skipping


## 7. Evaluation

Validation cross-entropy stratified by how often each code occurs in the training file actually used. The GRASP claim predicts that any gain concentrates in the rare strata.

Three details. Frequency strata come from `train.bin` itself, remembering that `utils.get_batch` shifts every raw token id by +1; the label file's counts describe the real UK Biobank cohort, whereas the model is trained on the synthetic release, where 327 codes never occur and all nine lifestyle tokens are absent, so there is an explicit unseen bin. Only targets Delphi is trained to predict are scored: `validation_loss_mode=True` sets the `ignore_tokens` logits to minus infinity and renormalises, exactly as `estimate_loss()` in `train.py` does, and those tokens plus "no event" are dropped from the targets. Batching mirrors `estimate_loss()`: `select='left'`, `no_event_token_rate=5`, `cut_batch=True`.

Batches are drawn from one fixed RNG, so every run sees identical targets. That makes the bootstrap paired: it resamples validation batches, not runs.

In [ ]:
import numpy as np, pandas as pd, torch
from model_untied import Delphi, DelphiConfig   # NOT `model` — plain DelphiConfig would
from utils import get_batch, get_p2i           # reject the tie_weights key in model_args

DATASET, DEVICE = 'ukb_simulated_data', 'cuda'
N_BATCHES, BATCH_SIZE, EVAL_SEED = 100, 128, 1234
SELECT = 'left'   # matches estimate_loss() in train.py. 'random' samples windows from anywhere in
                  # the trajectory instead of the first block_size tokens — worth rerunning with
                  # it as a robustness check, since 'left' is biased towards early-life events.

train_bin = np.fromfile(f'data/{DATASET}/train.bin', dtype=np.uint32).reshape(-1, 3)
val_bin   = np.fromfile(f'data/{DATASET}/val.bin',   dtype=np.uint32).reshape(-1, 3)
val_p2i   = get_p2i(val_bin)

# raw token id t in the .bin maps to MODEL token id t+1 (utils.get_batch: `tokens = tokens + 1`)
train_freq = np.bincount(train_bin[:, 2].astype(np.int64) + 1, minlength=V)[:V]
print(f"{(train_freq > 0).sum()} of {V} tokens occur in training; "
      f"{int((train_freq[13:] == 0).sum())} disease codes never occur")

def eval_run(out_dir):
    'Per-token NLL, split by validation batch so the bootstrap can resample batches.'
    ck = torch.load(f'{out_dir}/ckpt.pt', map_location=DEVICE, weights_only=False)
    a  = ck['model_args']
    m  = Delphi(DelphiConfig(**a))
    m.load_state_dict({k.replace('_orig_mod.', ''): v for k, v in ck['model'].items()})
    m.eval().to(DEVICE)

    ignored = sorted(set(a['ignore_tokens']) | {1})     # + 'no event', as validation_loss_mode does
    bt = np.zeros((N_BATCHES, V)); bc = np.zeros((N_BATCHES, V))
    rng = np.random.default_rng(EVAL_SEED)              # same batches for every run => paired
    for b in range(N_BATCHES):
        ix = rng.integers(0, len(val_p2i), BATCH_SIZE)
        X, A, Y, Bg = get_batch(ix, val_bin, val_p2i, block_size=a['block_size'], device=DEVICE,
                                select=SELECT, no_event_token_rate=5, cut_batch=True)
        with torch.no_grad():
            logits, _, _ = m(X, A, Y, Bg, validation_loss_mode=True)
        lp = torch.log_softmax(logits.float(), dim=-1)
        keepm = (Y != -1)
        for k in ignored:
            keepm &= (Y != k)
        yv  = Y[keepm]
        nll = -lp[keepm].gather(1, yv[:, None]).squeeze(1)
        assert torch.isfinite(nll).all(), "non-finite NLL — a scored target was masked to -inf"
        y = yv.cpu().numpy(); n = nll.cpu().numpy()
        np.add.at(bt[b], y, n); np.add.at(bc[b], y, 1)
    del m; torch.cuda.empty_cache()
    return bt, bc, int(ck['iter_num'])

per_run, ckpt_iter = {}, {}
for tie, arms in RUNS.items():
    for arm, dirs in arms.items():
        for d in dirs:
            bt, bc, it = eval_run(d)
            per_run[d], ckpt_iter[d] = (bt, bc), it
            print(f"{d:26s} best-val ckpt @ iter {it:>5d}  "
                  f"scored {int(bc.sum()):>7d} targets  mean NLL {bt.sum()/bc.sum():.4f}")

# the demo config has always_save_checkpoint=False, so ckpt.pt is the BEST-val checkpoint,
# not the last one. Arms can early-stop at different iterations — worth stating.
print("\ncheckpoint iterations differ across arms:", len(set(ckpt_iter.values())) > 1)

932 of 1270 tokens occur in training; 327 disease codes never occur
number of parameters: 2.24M
run_baseline_tied_s0       best-val ckpt @ iter  1250  scored  311024 targets  mean NLL 5.2923
number of parameters: 2.24M
run_baseline_tied_s1       best-val ckpt @ iter  1750  scored  311024 targets  mean NLL 5.2567
number of parameters: 2.24M
run_baseline_tied_s2       best-val ckpt @ iter  2000  scored  311024 targets  mean NLL 5.2640
number of parameters: 2.24M
run_control_tied_s0        best-val ckpt @ iter  2000  scored  311024 targets  mean NLL 5.2663
number of parameters: 2.24M
run_control_tied_s1        best-val ckpt @ iter  2000  scored  311024 targets  mean NLL 5.2625
number of parameters: 2.24M
run_control_tied_s2        best-val ckpt @ iter  1750  scored  311024 targets  mean NLL 5.2637
number of parameters: 2.24M
run_sem_tied_s0            best-val ckpt @ iter  2000  scored  311024 targets  mean NLL 5.2529
number of parameters: 2.24M
run_sem_tied_s1            best-val ckpt @ 

In [ ]:
counts = [c for _, c in per_run.values()]
assert all(np.array_equal(counts[0], c) for c in counts), "targets differ across runs — not paired"
BC = counts[0]        # identical for every run, so tied/untied are paired with each other too

# pool seeds within an arm: identical denominators, so averaging numerators averages the NLL
BT = {tie: {arm: np.mean([per_run[d][0] for d in dirs], axis=0) for arm, dirs in arms.items()}
      for tie, arms in RUNS.items()}

IGNORED = set(range(13))
BINS = [(0, 1, 'unseen'), (1, 10, 'ultra-rare'), (10, 100, 'very rare'),
        (100, 1000, 'rare'), (1000, 10**9, 'common')]
N_BOOT, BASE = 2000, 'baseline'
boot_rng = np.random.default_rng(7)
draws = boot_rng.integers(0, N_BATCHES, (N_BOOT, N_BATCHES))

def paired_ci(num, den):
    'Cluster bootstrap over validation batches. num/den are per-batch sums.'
    nb_, db_ = num[draws].sum(1), den[draws].sum(1)
    ok = db_ > 0                                    # a sparse bin can resample to zero events
    if den.sum() < 30 or ok.mean() < 0.95:
        return 'n/a (too few events)'
    lo_, hi_ = np.percentile(nb_[ok] / db_[ok], [2.5, 97.5])
    return f"[{lo_:+.3f}, {hi_:+.3f}]"

rows = []
for tie, arms in RUNS.items():
    for lo, hi, name in BINS:
        sel = (train_freq >= lo) & (train_freq < hi)
        sel[list(IGNORED)] = False
        den = BC[:, sel].sum(1)
        row = {'tie': tie, 'bin': name, 'n_codes': int(sel.sum()), 'n_val_events': int(den.sum())}
        if den.sum() == 0:
            rows.append(row); continue
        for arm in arms:
            row[arm] = round(float(BT[tie][arm][:, sel].sum() / den.sum()), 4)
        for arm in arms:
            if arm == BASE:
                continue
            num = (BT[tie][arm] - BT[tie][BASE])[:, sel].sum(1)
            row[f'Δ_{arm}']  = round(float(num.sum() / den.sum()), 4)
            row[f'CI_{arm}'] = paired_ci(num, den)
        # spread across seeds, so a difference can be read against run-to-run noise
        for arm, dirs in arms.items():
            row[f'sd_{arm}'] = round(float(np.std([per_run[d][0][:, sel].sum() / den.sum()
                                                   for d in dirs])), 4)
        rows.append(row)

tab = pd.DataFrame(rows)
tab.to_csv('stratified_loss.csv', index=False); keep('stratified_loss.csv')
for tie in RUNS:
    print(f"\n=== {tie} ===")
    print(tab[tab.tie == tie].drop(columns='tie').to_string(index=False))

saved -> /content/drive/MyDrive/delphi_5685/stratified_loss.csv

=== tied ===
       bin  n_codes  n_val_events  baseline  control     sem  Δ_control       CI_control   Δ_sem           CI_sem  sd_baseline  sd_control  sd_sem
    unseen      327           169   16.5769  17.7228 17.3865     1.1459 [+1.108, +1.186]  0.8096 [+0.762, +0.857]       1.4971      0.5152  1.4634
ultra-rare      308          2346   10.5069  10.6138 10.6125     0.1069 [+0.078, +0.138]  0.1056 [+0.078, +0.133]       0.1644      0.0172  0.2352
 very rare      334         21729    7.7399   7.7629  7.7591     0.0230 [+0.018, +0.029]  0.0192 [+0.014, +0.025]       0.0615      0.0041  0.0510
      rare      241        146185    5.7316   5.7350  5.7166     0.0034 [+0.002, +0.005] -0.0149 [-0.016, -0.013]       0.0142      0.0085  0.0109
    common       47        140595    4.3096   4.2842  4.2974    -0.0255 [-0.027, -0.024] -0.0122 [-0.013, -0.011]       0.0514      0.0084  0.0163

=== untied ===
       bin  n_codes  n_v

### The interaction test

Does semantic initialisation help more once the decoder is decoupled? That is the semantic arm's delta in the untied condition minus its delta in the tied condition, per bin, with a paired bootstrap over the same validation batches. A negative value would mean untying released an effect that tying was suppressing.

In [ ]:
rows = []
for lo, hi, name in BINS:
    sel = (train_freq >= lo) & (train_freq < hi)
    sel[list(IGNORED)] = False
    den = BC[:, sel].sum(1)
    if den.sum() == 0:
        continue
    d_un = (BT['untied']['sem'] - BT['untied'][BASE])[:, sel].sum(1)
    d_ti = (BT['tied'  ]['sem'] - BT['tied'  ][BASE])[:, sel].sum(1)
    diff = d_un - d_ti
    rows.append({'bin': name, 'n_val_events': int(den.sum()),
                 'Δ_sem tied':   round(float(d_ti.sum() / den.sum()), 4),
                 'Δ_sem untied': round(float(d_un.sum() / den.sum()), 4),
                 'untied − tied': round(float(diff.sum() / den.sum()), 4),
                 'CI': paired_ci(diff, den)})

interaction = pd.DataFrame(rows)
interaction.to_csv('tying_interaction.csv', index=False); keep('tying_interaction.csv')
print(interaction.to_string(index=False))

saved -> /content/drive/MyDrive/delphi_5685/tying_interaction.csv
       bin  n_val_events  Δ_sem tied  Δ_sem untied  untied − tied               CI
    unseen           169      0.8096        0.7573        -0.0523 [-0.096, -0.006]
ultra-rare          2346      0.1056        0.0676        -0.0380 [-0.078, -0.003]
 very rare         21729      0.0192        0.0244         0.0052 [-0.002, +0.012]
      rare        146185     -0.0149       -0.0011         0.0139 [+0.012, +0.016]
    common        140595     -0.0122        0.0078         0.0200 [+0.019, +0.021]


Reading the tables. A negative delta means lower cross-entropy, so better. A confidence interval that straddles zero means the arm is not distinguishable from baseline on this data; compare the delta against the spread across seeds before drawing a conclusion. Read within a tying condition, since the untied model carries 152K extra parameters and its absolute losses are not comparable to the tied model.

Three readings are possible within a condition. If the semantic delta is clearly negative in the rare bins and near zero in the common bin, with the control delta near zero throughout, semantic structure substitutes for training examples where examples are scarce. If the semantic and control deltas are similar and both non-zero, the effect is the initialisation geometry and not meaning; this is what the control arm exists to catch. If both are near zero with tight intervals, the result is negative at this scale on this data.

Caveats. The cohort is the authors' synthetic release, which they recommend against using for research, and the demo schedule is 5,000 iterations rather than the published run. In the tied condition "semantic input embeddings help" and "semantic output priors help" are not separable; the untied condition disentangles them at the cost of 152K parameters. The unseen bin is under-powered: the synthetic validation file contains only about 96 events on codes absent from training, so its interval stays wide however many batches are drawn. It is the sharpest test of the GRASP claim and the clearest reason to run this on real data.

In [ ]:
import json, datetime
ck = torch.load(f"{RUNS['tied'][BASE][0]}/ckpt.pt", map_location='cpu', weights_only=False)
a  = ck['model_args']
n_par = {t: sum(p.numel() for p in Delphi(DelphiConfig(
             **{**a, 'tie_weights': t == 'tied'})).parameters()) for t in RUNS}

summary = {
    'date': str(datetime.date.today()),
    'design': '3 init arms x {tied, untied} x seeds',
    'runs': RUNS,
    'seeds': SEEDS,
    'n_parameters': n_par,
    'embedding_backend': BACKEND,
    'embedding_model': LOCAL_MODEL if BACKEND == 'local' else 'models/gemini-embedding-001',
    'embedding_dim_raw': int(E.shape[1]),
    'projected_dim': N_EMBD,
    'init_std': INIT_STD,
    'init_sha256': SHA,          # identical across tied and untied by construction
    'pca_var_explained': round(PCA_VAR, 4) if PCA_VAR else 'reused from v2',
    'chapter_purity_at10': round(float(purity), 4),
    'chapter_purity_null': round(float(null), 4),
    'model_args': {k: v for k, v in a.items() if k != 'ignore_tokens'},
    'train_iters': int(ck['iter_num']),
    'eval': {'n_batches': N_BATCHES, 'batch_size': BATCH_SIZE, 'seed': EVAL_SEED,
             'select': SELECT, 'scored_targets': int(BC.sum()),
             'excluded_tokens': 'ignore_tokens + no-event (validation_loss_mode)'},
    'data': f'public synthetic {DATASET} (authors: code-testing only, not for research claims)',
    'interaction': interaction.to_dict('records'),
    'note_weight_tying': 'tie_weights added to DelphiConfig in model_untied.py; untied adds '
                         'vocab_size*n_embd params, so compare within a condition',
    'note_frozen': 'frozen wte in the tied model also freezes lm_head, so no frozen arm is in '
                   'the comparison; frozen+untied is left as future work',
}
json.dump(summary, open('run_summary.json', 'w'), indent=2)
keep('run_summary.json')
print(json.dumps(summary, indent=2))

number of parameters: 2.24M
number of parameters: 2.40M
saved -> /content/drive/MyDrive/delphi_5685/run_summary.json
{
  "date": "2026-08-09",
  "design": "3 init arms x {tied, untied} x seeds",
  "runs": {
    "tied": {
      "baseline": [
        "run_baseline_tied_s0",
        "run_baseline_tied_s1",
        "run_baseline_tied_s2"
      ],
      "control": [
        "run_control_tied_s0",
        "run_control_tied_s1",
        "run_control_tied_s2"
      ],
      "sem": [
        "run_sem_tied_s0",
        "run_sem_tied_s1",
        "run_sem_tied_s2"
      ]
    },
    "untied": {
      "baseline": [
        "run_baseline_untied_s0",
        "run_baseline_untied_s1",
        "run_baseline_untied_s2"
      ],
      "control": [
        "run_control_untied_s0",
        "run_control_untied_s1",
        "run_control_untied_s2"
      ],
      "sem": [
        "run_sem_untied_s0",
        "run_sem_untied_s1",
        "run_sem_untied_s2"
      ]
    }
  },
  "seeds": [
    0,
    1,
    2
